# 05 同じ合成runを自己完結HTMLに書き出す

## Goal

共通サービスの DemoRun を reports.render_demo_report に渡し、資産指数・ドローダウン・出典・費用を記録したHTMLを作ります。レポートAPIは渡されたrunを描画し、取得や再バックテストをしません。

## Setup

- 入力: 固定合成デモのrun IDとSHA-256。USD / raw、判断時刻UTC
- 評価: 終値から次区間へ1期遅れの配分。手数料3bps、スリッページ2bps
- 03の単一銘柄OOS比較とは別の2銘柄デモrunです。結果の数値を直接比較しません
- 出力先: WSLの一時ディレクトリ。Git管理下へHTMLや口座データを書きません
- 外部通信はしません。HTMLはCSS内蔵で外部のscript、画像、stylesheetを読みません

In [ ]:
from pathlib import Path
from html.parser import HTMLParser
from tempfile import gettempdir

from market_research.reports import render_demo_report
from market_research.services import build_demo_run

run = build_demo_run()
print(f"run_id={run.run_id}, input_sha256={run.input_hash}, mode={run.mode}")

## Steps

### 1. 既存runをHTMLにする

report_html はファイルへ書く前にメモリ上で検証できます。run IDと入力ハッシュで再現元を特定します。

In [ ]:
report_html = render_demo_report(run)
assert report_html.startswith("<!doctype html>")
assert run.run_id in report_html and run.input_hash in report_html
assert "<style>" in report_html
class TagCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.names = set()

    def handle_starttag(self, tag, attrs):
        self.names.add(tag)


tags = TagCollector()
tags.feed(report_html)
assert {"script", "link", "iframe", "img"}.isdisjoint(tags.names)
print(f"HTML chars={len(report_html)}, run_id={run.run_id}")

### 2. Git管理外の一時領域へ保存する

表示されたWSLパスをブラウザーで開いてください。実データに使う場合も出力先と公開範囲を明示してから保存します。

In [ ]:
report_path = Path(gettempdir()) / f"market-research-demo-{run.run_id}.html"
report_path.write_text(report_html, encoding="utf-8")
print(f"saved={report_path.resolve()}, bytes={report_path.stat().st_size}")

## Checks

表示内容は合成runからの計算です。元のrunとHTMLに同じ識別子が含まれること、出力ファイルがリポジトリ外にあることを確認します。実約定・FX・税務は対象外です。

In [ ]:
assert report_path.is_file()
assert report_path.read_text(encoding="utf-8") == report_html
assert "合成デモ" in report_html
assert "1期遅れ" in report_html
print("HTMLの保存と入力識別子の一致を確認")

## Next Steps

HTMLをブラウザーで確認してください。再実行すると同じrun IDの一時ファイルを更新します。保存データのレポート作成は別の入口・契約です。